In [1]:
# Report on data quality for top 50 sample

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd

load_dotenv(find_dotenv())

USABILITY_THRESHOLD = 0.85  # threshold of data availability to be considered usable

STATUSES = ['Active', 'Final', 'Inactive', 'In Review']
DATE_CONCEPTS = ['FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE', 'PERMIT_OR_FILE_DATE']

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

JURS_FILEPATH = os.path.join(MY_DATA_PATH, "raw_data", "dewey_la_county_jurisdictions.csv")
REPORT_FILEPATH = os.path.join(ROOT_PATH, "reports", "2026-07-22-quality-report-la-sample.md")
INPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "permits_la_sample.parquet")

In [2]:
# Load the data

city_df = pd.read_csv(JURS_FILEPATH)
df = pd.read_parquet(INPUT_FILEPATH)

df['SCHEMA'] = df['DATA'].apply(du.detect_schema)

In [3]:
MD = f"""
# Building Permits Data Usability Report

For the jurisdictions in LA County, random sample of 2000 permits per city.  The jurisdctions are:

{city_df['JURISDICTION'].unique().tolist()}

Key variables:
- FILE_DATE: Application date of permit
- PERMIT_DATE: Issue date of permit
- FINAL_DATE: Final date of permit
- PERMIT_OR_FILE_DATE: PERMIT_DATE if available, FILE_DATE otherwise
- STATUS_NORMALIZED: Normalized status of permit: "Active", "Final", "Inactive", "In Review"

Notes:
- This report checks the availability of the key date fields by the status of the permit and reports on the number of cities with usable data.
- This report does not de-duplicate the data. It is known that there are permit duplicates, but we have not yet investigated the extent of the problem.
- This report matches each city to a single jurisdiction in the permits data. It is possible that one city or metro area could be served by multiple jurisdictions (a jurisdiction is a building permitting authority). Thus, in real work, we'd likely want to match permits to cities based on the geocoded addresses, not on the name of the jurisdiction.
- Nuances regarding the correct interpretation of FILE_DATE, PERMIT_DATE, and FINAL_DATE may depend on the jurisdiction and warrants further investigation.
- Reported date ranges are based on 1st and 99th percentiles to avoid outliers (which are likely recording errors).


"""

In [4]:
# Data quality report

MD += du.data_quality_report(df, threshold=USABILITY_THRESHOLD)

In [5]:
CONCLUSION = """

## Conclusion

- PERMIT_OR_FILE_DATE appears mostly usable. 
- FINAL_DATE appears usable only for some jurisdictions. 
- Need to investigate:
    - If and when PERMIT_DATE and FILE_DATE are interchangeable
    - If and when FINAL_DATE is available
    - Why certain dates are not available for certain jurisdictions

"""

with open(REPORT_FILEPATH, 'w') as f:
    f.write(MD + CONCLUSION)